<a href="https://colab.research.google.com/github/JoseORHub/Alertas_Invima/blob/main/Scrapping_Alertas_Invima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# INVIMA - Scraper de alertas de medicamentos
# ============================================================
#   1. Ejecuta la celda de instalación primero
# ============================================================

!pip install -q pandas openpyxl beautifulsoup4 lxml pymupdf requests pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 11.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 2. Luego ejecuta este script
# ============================================================

import re
import io
import time
from datetime import datetime
from urllib.parse import urljoin

import requests
from pypdf import PdfReader
import pandas as pd
from bs4 import BeautifulSoup

# ============================================================
# CONFIGURACIÓN
# ============================================================

BASE_URL = "https://app.invima.gov.co/alertas/medicamentos-productos-biologicos"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "es-CO,es;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Referer": BASE_URL,
}

REQUEST_TIMEOUT = 60
PDF_TIMEOUT = 90
PAUSE_BETWEEN_PAGES = 0.6
PAUSE_BETWEEN_PDFS = 0.1
MAX_RETRIES = 3
RETRY_WAIT = 2

SPANISH_MONTHS = (
    r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|"
    r"septiembre|setiembre|octubre|noviembre|diciembre)"
)
RE_DATE   = re.compile(rf"(?i)\bBogotá,?\s*\d{{1,2}}\s+(?:de\s+)?{SPANISH_MONTHS}\s+(?:de\s+)?\d{{4}}\b")
RE_ALERT  = re.compile(r"(?i)\b(Alerta|Informe\s+de\s+Seguridad)\s+No\.?\s*#?\s*([A-Za-z0-9\-]+)")
RE_RISARH = re.compile(r"\b(MA\d{3,}-\d+|\d{4}-\d{4}-\d{4})\b", re.IGNORECASE)
RE_SKIP   = re.compile(r"(?i)alerta\s+no|bogotá|invima|subdirección|dirección|notificación|fecha|radicado|asunto")
RE_INST   = re.compile(r"(?i)república|invima|ministerio|instituto|dirección|subdirección")

CSV_COLUMNS = [
    "Fecha de notificación",
    "Número de la alerta",
    "Nombre del producto",
    "RISARH",
    "Dirección de enlace",
]

# ============================================================
# SCRAPER — peticiones HTTP
# ============================================================

def get_page_html(page_index: int) -> str:
    url = BASE_URL if page_index == 0 else f"{BASE_URL}?page={page_index}"
    response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.text


def find_pdf_links(html: str) -> list:
    soup = BeautifulSoup(html, "lxml")
    links = [
        urljoin(BASE_URL, a.get("href"))
        for a in soup.find_all("a")
        if (a.get_text() or "").strip().lower() == "ver" and a.get("href")
    ]
    pdf_links = [u for u in links if ".pdf" in u.lower()]
    return list(dict.fromkeys(pdf_links))  # elimina duplicados


def fetch_pdf_bytes(url: str, session: requests.Session) -> bytes:
    for attempt in range(1, MAX_RETRIES + 1):
        response = session.get(url, headers=HEADERS, timeout=PDF_TIMEOUT)
        if response.status_code == 200 and response.content[:4] == b'%PDF':
            return response.content
        print(f"    Intento {attempt}/{MAX_RETRIES} — HTTP {response.status_code}")
        if attempt < MAX_RETRIES:
            time.sleep(RETRY_WAIT * attempt)
    raise Exception(f"HTTP {response.status_code} tras {MAX_RETRIES} intentos: {url}")

# ============================================================
# PARSER — extracción de datos desde PDF
# ============================================================

def _empty_record(pdf_url: str) -> dict:
    return {col: "" for col in CSV_COLUMNS} | {"Dirección de enlace": pdf_url}


def _extract_alert_number(text: str) -> str:
    match = RE_ALERT.search(text)
    if not match:
        return ""
    tipo   = re.sub(r"\s+", " ", match.group(1)).strip()  # "Alerta" o "Informe de Seguridad"
    numero = match.group(2).strip()
    return f"{tipo} No. {numero}".replace("No. No.", "No.")


def _extract_date(text: str) -> str:
    match = RE_DATE.search(text)
    if not match:
        return ""
    cleaned = re.sub(r"Bogotá\s*", "Bogotá", match.group(0), flags=re.IGNORECASE)
    return re.sub(r"\s{2,}", " ", cleaned).strip()


RE_NOMBRE = re.compile(r"(?i)^Nombre\s+del\s+producto\s*:\s*(.+)$")
RE_SKIP_TITLE = re.compile(r"(?i)^(invima\s+alerta|alerta\s+sanitaria|dirección\s+de)$")

def _extract_product_name(lines: list, date_match) -> str:
    # Estrategia 1: campo explícito "Nombre del producto: ..."
    for line in lines:
        m = RE_NOMBRE.match(line)
        if m and m.group(1).strip():
            return m.group(1).strip()

    # Estrategia 2: tras la fecha, bloque de líneas en mayúsculas (título largo)
    if date_match:
        date_norm = date_match.group(0).lower()
        date_idx = next((i for i, ln in enumerate(lines) if date_norm in ln.lower()), None)
        if date_idx is not None:
            title_parts = []
            for line in lines[date_idx + 1: date_idx + 15]:
                if RE_SKIP.search(line) or RE_SKIP_TITLE.match(line):
                    continue
                if line.isupper() and len(line) >= 6:
                    title_parts.append(line)
                elif title_parts:
                    break  # fin del bloque en mayúsculas
                elif len(line) >= 6:
                    return line  # primera línea mixta válida
            if title_parts:
                return " ".join(title_parts)

    # Fallback: primeras líneas con forma de título
    for line in lines[:20]:
        if (line.isupper() or line == line.title()) and 5 < len(line) < 160:
            if not RE_INST.search(line) and not RE_SKIP_TITLE.match(line):
                return line
    return ""


def extract_from_pdf(pdf_bytes: bytes, pdf_url: str) -> dict:
    record = _empty_record(pdf_url)
    try:
        reader = PdfReader(io.BytesIO(pdf_bytes))
        if not reader.pages:
            return record
        text  = (reader.pages[0].extract_text() or "").strip()
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        date_match = RE_DATE.search(text)

        record["Número de la alerta"]   = _extract_alert_number(text)
        record["Fecha de notificación"] = _extract_date(text)
        record["Nombre del producto"]   = _extract_product_name(lines, date_match)
        record["RISARH"]                = (m := RE_RISARH.search(text)) and m.group(0).upper() or ""
    except Exception:
        pass
    return record

# ============================================================
# RUNNER — orquesta el flujo
# ============================================================

def scrape(n_pages: int = 1) -> list:
    records, seen = [], set()
    session = requests.Session()
    session.get(BASE_URL, headers=HEADERS, timeout=REQUEST_TIMEOUT)  # establece cookies
    for page in range(n_pages):
        print(f"📄 Procesando página {page + 1} de {n_pages}...")
        html = get_page_html(page)
        for url in find_pdf_links(html):
            if url in seen:
                continue
            seen.add(url)
            try:
                record = extract_from_pdf(fetch_pdf_bytes(url, session), url)
            except Exception as e:
                print(f"  ⚠️  Error en {url}:\n      {e}")
                record = _empty_record(url)
            records.append(record)
            time.sleep(PAUSE_BETWEEN_PDFS)
        time.sleep(PAUSE_BETWEEN_PAGES)
    return records

# ============================================================
# EXPORTADOR — guarda el CSV
# ============================================================

def export_to_csv(records: list, filename: str = None) -> str:
    if not records:
        print("No se extrajo información.")
        return ""
    filepath = filename or f"alertas_invima_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    pd.DataFrame.from_records(records, columns=CSV_COLUMNS).to_csv(
        filepath, index=False, encoding="utf-8-sig"
    )
    print(f"✅ CSV guardado: {filepath}  ({len(records)} registros)")
    return filepath

# ============================================================
# EJECUCIÓN
# ============================================================

try:
    n_pages = int(input("¿Cuántas páginas deseas inspeccionar? (ej. 3): ").strip())
except ValueError:
    print("Entrada inválida. Usando 1 página.")
    n_pages = 1

records = scrape(n_pages=n_pages)
export_to_csv(records)

¿Cuántas páginas deseas inspeccionar? (ej. 3): 1
📄 Procesando página 1 de 1...
✅ CSV guardado: alertas_invima_20260605_132647.csv  (8 registros)


'alertas_invima_20260605_132647.csv'